In [5]:
#q1)


import numpy as np, pandas as pd
from scipy import stats
np.random.seed(42)

def moct_np(a):
    a = np.asarray(a)
    return dict(mean=np.mean(a), median=np.median(a), mode=stats.mode(a, keepdims=False).mode)

# ---- Q1: discrete & continuous, 1D & 2D ----
print("=== Q1 ===")
disc = np.random.randint(1, 11, 30)          # discrete
cont = np.round(np.random.normal(50, 10, 30), 2)  # continuous
print("Discrete 1D :", disc); print(moct_np(disc))
print("Continuous 1D:", cont[:10], "..."); print(moct_np(cont))
s = pd.Series(disc)
print("Pandas Series -> mean %.2f median %.2f mode %s" % (s.mean(), s.median(), s.mode().tolist()))

d2 = np.random.randint(1, 11, (5, 4))
c2 = np.round(np.random.normal(50, 10, (5, 4)), 2)
print("\n2D discrete:\n", d2)
print("column mean", d2.mean(axis=0), "row mean", d2.mean(axis=1))
print("\ncolumn median", np.median(d2, axis=0))
print("\ncolumn mode", stats.mode(d2, axis=0, keepdims=False).mode)
df = pd.DataFrame(c2, columns=list("ABCD"))
print("\n2D continuous DataFrame:\n", df)
print("mean\n", df.mean(), "\nmedian\n", df.median(), "\nmode\n", df.mode().iloc[0])

# ---- Q2: outliers ----
print("\n=== Q2 Outlier effect ===")
base = np.array([10, 12, 12, 13, 14, 15, 15, 15, 16, 18])
out = np.append(base, 200)
for name, a in [("without outlier", base), ("with outlier 200", out)]:
    print(f"{name:18s}", {k: round(float(v), 2) for k, v in moct_np(a).items()})
print("Observation: mean shifts hugely, median shifts slightly, mode unchanged -> median/mode are robust.")

# ---- Q3: sample dataset 50-100 instances ----
print("\n=== Q3 Sample dataset (80 instances) ===")
n = 80
data = pd.DataFrame({
    "Age": np.random.randint(18, 60, n),                       # discrete
    "Salary": np.round(np.random.normal(45000, 8000, n), 2),   # continuous
    "Score": np.random.randint(40, 100, n)})
print(data.head())
print("Mean\n", data.mean(), "\nMedian\n", data.median(), "\nMode\n", data.mode().iloc[0])
data2 = data.copy(); data2.loc[0, "Salary"] = 500000          # outlier
print("\nSalary with outlier -> mean %.2f (was %.2f), median %.2f (was %.2f)" %
      (data2.Salary.mean(), data.Salary.mean(), data2.Salary.median(), data.Salary.median()))

=== Q1 ===
Discrete 1D : [ 7  4  8  5  7 10  3  7  8  5  4  8  8  3  6  5  2  8  6  2  5  1 10  6
  9  1 10  3  7  4]
{'mean': np.float64(5.733333333333333), 'median': np.float64(6.0), 'mode': np.int32(8)}
Continuous 1D: [44.56 51.11 38.49 53.76 43.99 47.08 43.98 68.52 49.87 39.42] ...
{'mean': np.float64(47.38066666666667), 'median': np.float64(47.035), 'mode': np.float64(30.4)}
Pandas Series -> mean 5.73 median 6.00 mode [8]

2D discrete:
 [[ 8  3  1  8]
 [ 3  3  1  5]
 [10  7 10  9]
 [ 7  9  8  2]
 [ 1  7  7  8]]
column mean [5.8 5.8 5.4 6.4] row mean [5.   3.   9.   6.5  5.75]

column median [7. 7. 7. 8.]

column mode [1 3 1 8]

2D continuous DataFrame:
        A      B      C      D
0  59.50  67.27  54.58  33.16
1  53.27  49.19  54.68  57.36
2  42.20  41.56  48.49  40.34
3  51.50  48.87  76.34  39.75
4  42.18  54.24  58.73  72.87
mean
 A    49.730
B    52.226
C    58.564
D    48.696
dtype: float64 
median
 A    51.50
B    49.19
C    54.68
D    40.34
dtype: float64 
mode
 A    42.1

In [ ]:
#q2)

"""Experiment 2: Relationship among Mean/Median/Mode & effect of transformations"""
import numpy as np
from scipy import stats
np.random.seed(1)

def mode_cont(a, bins=60):               # mode of continuous data = peak of histogram
    h, e = np.histogram(a, bins=bins); i = h.argmax()
    return (e[i] + e[i+1]) / 2
def report(name, a):
    m, md, mo = a.mean(), np.median(a), mode_cont(a)
    print(f"{name:16s} mean={m:8.3f} median={md:8.3f} mode={mo:8.3f} skew={stats.skew(a):6.3f}")
    return m, md, mo

print("=== Q1 ===")
sym   = np.random.normal(50, 5, 100000)
right = np.random.gamma(2, 5, 100000)               # right skewed
left  = 100 - np.random.gamma(2, 5, 100000)         # left skewed
a = report("Symmetric", sym);   print("  -> Mean ≈ Median ≈ Mode:", np.allclose(a, a[0], atol=0.5))
b = report("Right skewed", right); print("  -> Mode < Median < Mean:", b[2] < b[1] < b[0])
c = report("Left skewed", left);   print("  -> Mean < Median < Mode:", c[0] < c[1] < c[2])

print("\n=== Q2 Transformations ===")
x = np.array([2, 3, 3, 5, 7, 8, 3, 9, 10])
def mm(a): return a.mean(), np.median(a), stats.mode(a, keepdims=False).mode
print("original     ", mm(x))
print("shift  (x+10)", mm(x + 10), "-> all MOCT shift by +10")
print("scale  (x*3) ", mm(x * 3),  "-> all MOCT scale by 3")
print("mix (2x+5)   ", mm(2 * x + 5), "-> new = 2*old + 5 for each")
o = mm(x); n = mm(2 * x + 5)
print("check:", all(np.isclose(2 * o[i] + 5, n[i]) for i in range(3)))

In [ ]:
#q3)

"""Experiment 3: Measures of Spread & Quartiles"""
import numpy as np
np.random.seed(7)

def spread(a):
    a = np.asarray(a, float)
    return dict(range=a.max() - a.min(), variance=a.var(ddof=1), std=a.std(ddof=1))

print("=== Q1 Spread ===")
data = np.array([12, 15, 14, 10, 18, 20, 22, 16, 15, 13, 17, 19])
print(spread(data))

print("\n=== Q2 Outliers ===")
out = np.append(data, 100)
print("without:", {k: round(v, 2) for k, v in spread(data).items()})
print("with   :", {k: round(v, 2) for k, v in spread(out).items()})

print("\n=== Q3 Xnew = c*Xold + d ===")
c, d = 3, 10
new = c * data + d
s0, s1 = spread(data), spread(new)
print("old", {k: round(v, 2) for k, v in s0.items()})
print("new", {k: round(v, 2) for k, v in s1.items()})
print(f"range x{s1['range']/s0['range']:.1f} (=|c|), std x{s1['std']/s0['std']:.1f} (=|c|), "
      f"var x{s1['variance']/s0['variance']:.1f} (=c^2); d has NO effect on spread")

print("\n=== Q4 Quartiles ===")
sample = np.sort(np.random.randint(10, 100, 40))
print("sorted:", sample)
def median(a):
    a = sorted(a); n = len(a)
    return a[n//2] if n % 2 else (a[n//2 - 1] + a[n//2]) / 2
n = len(sample)
Q2 = median(sample)
lower = sample[: n // 2]; upper = sample[(n + 1) // 2:]
Q1, Q3 = median(lower), median(upper)
IQR = Q3 - Q1
print(f"Q1={Q1}, Q2={Q2}, Q3={Q3}, IQR={IQR}")
print("numpy check:", np.percentile(sample, [25, 50, 75], method="midpoint"))
quart = {"Q1 (<=Q1)": sample[sample <= Q1],
         "Q2 (Q1<x<=Q2)": sample[(sample > Q1) & (sample <= Q2)],
         "Q3 (Q2<x<=Q3)": sample[(sample > Q2) & (sample <= Q3)],
         "Q4 (>Q3)": sample[sample > Q3]}
for k, v in quart.items(): print(f"{k:15s} count={len(v):2d} {v}")
lo, hi = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
print(f"Outlier fences: [{lo}, {hi}]  outliers: {sample[(sample<lo)|(sample>hi)]}")


In [ ]:
#q4a)

from math import sqrt

# a) Nominal: d = (p - m)/p  (p = #attributes, m = #matches)
def nominal_dissim(a, b):
    p = len(a); m = sum(x == y for x, y in zip(a, b))
    return (p - m) / p

# b) Binary: contingency q (1,1) r (1,0) s (0,1) t (0,0)
def _qrst(a, b):
    q = sum(x == 1 and y == 1 for x, y in zip(a, b)); r = sum(x == 1 and y == 0 for x, y in zip(a, b))
    s = sum(x == 0 and y == 1 for x, y in zip(a, b)); t = sum(x == 0 and y == 0 for x, y in zip(a, b))
    return q, r, s, t
def binary_symmetric(a, b):
    q, r, s, t = _qrst(a, b); return (r + s) / (q + r + s + t)
def binary_asymmetric(a, b):          # Jaccard dissimilarity (t ignored)
    q, r, s, t = _qrst(a, b); return (r + s) / (q + r + s)

# c) Numerical
def manhattan(a, b): return sum(abs(x - y) for x, y in zip(a, b))
def euclidean(a, b): return sqrt(sum((x - y) ** 2 for x, y in zip(a, b)))

# d) Ordinal: rank -> z = (r-1)/(M-1) -> Manhattan/Euclid on z
def ordinal_dissim(a, b, order):
    M = len(order)
    z = lambda v: (order.index(v)) / (M - 1)       # (rank-1)/(M-1), rank-1 == index
    return abs(z(a) - z(b))

# e) Cosine similarity
def cosine_sim(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    na, nb = sqrt(sum(x * x for x in a)), sqrt(sum(y * y for y in b))
    return dot / (na * nb)

print("Nominal  ['red','M','yes'] vs ['red','L','no'] :", round(nominal_dissim(['red','M','yes'], ['red','L','no']), 3))
x, y = [1, 0, 1, 1, 0, 0, 1], [1, 1, 0, 1, 0, 0, 0]
print("Binary   q,r,s,t =", _qrst(x, y))
print("  symmetric  d =", round(binary_symmetric(x, y), 3))
print("  asymmetric d =", round(binary_asymmetric(x, y), 3), "(Jaccard sim = %.3f)" % (1 - binary_asymmetric(x, y)))
p, q = [22, 1, 42, 10], [20, 0, 36, 8]
print("Numeric  Manhattan =", manhattan(p, q), " Euclidean =", round(euclidean(p, q), 3))
order = ['poor', 'fair', 'good', 'excellent']
print("Ordinal  'poor' vs 'excellent' d =", ordinal_dissim('poor', 'excellent', order),
      "| 'fair' vs 'good' d =", round(ordinal_dissim('fair', 'good', order), 3))
v1, v2 = [1, 1, 0, 0], [0, 1, 1, 0]
print("Cosine   sim =", round(cosine_sim(v1, v2), 3), " (angle-based, ignores magnitude)",
      "| parallel vectors:", round(cosine_sim([1, 2, 3], [2, 4, 6]), 3))


In [ ]:
#q4b)

"""Experiment 4(b): Mixed-type dissimilarity matrix
d(i,j) = sum_f( delta_ij^f * d_ij^f ) / sum_f( delta_ij^f )"""
import numpy as np, pandas as pd

# attribute types: nominal, ordinal, asymmetric binary x2, numerical
data = pd.DataFrame({
    "Colour":  ["red", "blue", "red", "green", "blue"],                 # nominal
    "Quality": ["fair", "good", "excellent", "fair", "good"],           # ordinal
    "Test1":   [1, 0, 1, 0, 0],                                         # asymmetric binary
    "Test2":   [0, 0, 1, 0, 0],                                         # asymmetric binary
    "Price":   [45.0, 22.0, 64.0, 28.0, 30.0]})                         # numerical
types = {"Colour": "nominal", "Quality": "ordinal", "Test1": "asym", "Test2": "asym", "Price": "numeric"}
order = {"Quality": ["fair", "good", "excellent"]}

def attr_dissim(col, a, b):
    t = types[col]
    if t == "nominal": return 0.0 if a == b else 1.0
    if t == "ordinal":
        M = len(order[col]); return abs(order[col].index(a) - order[col].index(b)) / (M - 1)
    if t == "asym": return 0.0 if a == b else 1.0
    if t == "numeric": return abs(a - b) / (data[col].max() - data[col].min())

def delta(col, a, b):                      # asymmetric binary: 0/0 match is ignored
    return 0 if (types[col] == "asym" and a == 0 and b == 0) else 1

def mixed(i, j):
    num = den = 0
    for col in data.columns:
        a, b = data.at[i, col], data.at[j, col]
        d = delta(col, a, b); num += d * attr_dissim(col, a, b); den += d
    return num / den if den else 0.0

n = len(data)
D = np.zeros((n, n))
for i in range(n):
    for j in range(n): D[i, j] = round(mixed(i, j), 3)
print(data, "\n")
print("Dissimilarity matrix:\n", pd.DataFrame(D, index=[f"O{i+1}" for i in range(n)], columns=[f"O{i+1}" for i in range(n)]))


In [ ]:
#Q5)

"""Experiment 5: Sampling techniques & choosing best sample"""
import numpy as np, math
np.random.seed(11)

# ---- Q1: sample size functions ----
def random_sample_size(N, Z=1.96, p=0.5, e=0.05):
    """Cochran + finite population correction"""
    n0 = (Z ** 2 * p * (1 - p)) / e ** 2
    return math.ceil(n0 / (1 + (n0 - 1) / N))
def cluster_sample_size(N, cluster_size, Z=1.96, p=0.5, e=0.05, deff=1.5):
    """Design effect inflates the SRS size; return (#individuals, #clusters)"""
    n = random_sample_size(N, Z, p, e) * deff
    return math.ceil(n), math.ceil(n / cluster_size)
def stratified_sample_size(strata_sizes, total_n):
    """Proportional allocation: n_h = n * N_h / N"""
    N = sum(strata_sizes)
    return [round(total_n * Nh / N) for Nh in strata_sizes]

print("=== Q1 ===")
print("Random     N=10000 ->", random_sample_size(10000))
print("Cluster    N=10000, cluster=50 -> (people, clusters) =", cluster_sample_size(10000, 50))
n = random_sample_size(10000)
print("Stratified strata=[5000,3000,2000], n=%d ->" % n, stratified_sample_size([5000, 3000, 2000], n))

# ---- Q2: best of k samples ----
print("\n=== Q2 ===")
N, k, n_ = 10000, 10, 200
pop = np.random.normal(100, 15, N)
pm, ps = pop.mean(), pop.std()
print(f"Population: mean={pm:.3f}, median={np.median(pop):.3f}, std={ps:.3f}")
rows = []
for i in range(k):
    s = np.random.choice(pop, n_, replace=False)
    # goodness = total deviation of MOCT (mean, median) and MOS (std) from population
    err = abs(s.mean() - pm) + abs(np.median(s) - np.median(pop)) + abs(s.std() - ps)
    rows.append((i + 1, s.mean(), np.median(s), s.std(), err))
    print(f"Sample {i+1:2d}: mean={s.mean():8.3f} median={np.median(s):8.3f} std={s.std():7.3f} error={err:.3f}")
best = min(rows, key=lambda r: r[-1])
print(f"\nBEST = Sample {best[0]} (lowest error {best[-1]:.3f})")


In [ ]:
#Q6)

"""Experiment 6: K-Means clustering from scratch (centroid based) + evaluation on Iris"""
import numpy as np, itertools
from sklearn.datasets import load_iris   # used ONLY to load data & for the SSE/silhouette check

class KMeans:
    def __init__(self, k=3, max_iter=300, tol=1e-6, seed=0):
        self.k, self.max_iter, self.tol, self.seed = k, max_iter, tol, seed
    def fit(self, X):
        rng = np.random.default_rng(self.seed)
        self.centroids = X[rng.choice(len(X), self.k, replace=False)]       # 1. random init
        for it in range(self.max_iter):
            d = np.linalg.norm(X[:, None] - self.centroids[None], axis=2)    # 2. distance to every centroid
            labels = d.argmin(axis=1)                                         #    assign to nearest
            new = np.array([X[labels == j].mean(axis=0) if (labels == j).any() else self.centroids[j]
                            for j in range(self.k)])                          # 3. update centroids (means)
            shift = np.linalg.norm(new - self.centroids)
            self.centroids = new
            if shift < self.tol: break                                        # 4. repeat until convergence
        self.labels_, self.n_iter_ = labels, it + 1
        self.inertia_ = ((X - self.centroids[labels]) ** 2).sum()             # SSE (WCSS)
        return self

def best_of(X, k, runs=10):               # random init can hit local minima -> restart, keep lowest SSE
    return min((KMeans(k, seed=s).fit(X) for s in range(runs)), key=lambda m: m.inertia_)

def cluster_accuracy(y, labels, k):       # clusters are unlabeled -> try all label permutations
    best = 0
    for perm in itertools.permutations(range(k)):
        mapped = np.array([perm[l] for l in labels]); best = max(best, (mapped == y).mean())
    return best

def silhouette(X, labels):
    D = np.linalg.norm(X[:, None] - X[None], axis=2); s = []
    for i in range(len(X)):
        same = labels == labels[i]; a = D[i][same & (np.arange(len(X)) != i)].mean()
        b = min(D[i][labels == c].mean() for c in set(labels) if c != labels[i])
        s.append((b - a) / max(a, b))
    return np.mean(s)

iris = load_iris(); X, y = iris.data, iris.target
m = best_of(X, 3)
print("Model: K-Means (k=3), Euclidean distance, 10 random restarts")
print("Iterations:", m.n_iter_, "| SSE/inertia: %.3f" % m.inertia_)
print("Centroids:\n", np.round(m.centroids, 3))
print("Cluster sizes:", np.bincount(m.labels_))
print("Accuracy (best cluster->class mapping): %.2f%%" % (100 * cluster_accuracy(y, m.labels_, 3)))
print("Silhouette score: %.3f" % silhouette(X, m.labels_))

print("\nElbow method (SSE vs k):")
for k in range(1, 8): print(f"  k={k}  SSE={best_of(X, k).inertia_:8.2f}")
